# LiDAR → radar transfer: what happened, where it breaks, what to try next

**Setup:** *Add Input* → your `dsert-analysis` dataset (the folder written by `analyze.sbatch` + `analyze_extra.sbatch`).
CPU is enough. Settings → Internet **On** (to fetch the analysis helpers from GitHub).

All feature files cover the **same labelled objects in the same order**, so models can be compared object by object.
Models (whichever are present):

| name | weights | input |
|---|---|---|
| `random_init_on_radar` / `random_init_on_livox` | untrained (seed 0) | radar / Livox |
| `radar_scratch` | trained on radar from random init | radar |
| `radar_ft_from_livox` | Livox-pretrained encoder, fine-tuned on radar | radar |
| `livox_e20_on_radar` | Livox model, **no** fine-tuning (zero-shot) | radar |
| `livox_pretrain` | Livox model | Livox |

In [ ]:
# 1) Locate the data and the analysis helpers
import os, glob, sys, zipfile, subprocess
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display, Image
except ImportError:
    display, Image = print, None

REPO_URL = "https://github.com/MuhammadIbneRafiq/DSERT-RoLL-3DOD.git"
D = os.environ.get("ANALYSIS_DIR")
if D is None:
    hits = glob.glob("/kaggle/input/**/feats_radar_scratch.npz", recursive=True)
    assert hits, "Add the dsert-analysis dataset as input"
    D = os.path.dirname(hits[0])

GC = os.path.join(D, "gradcam")                       # Grad-CAM images, one folder per weather
if not os.path.isdir(GC):
    zips = glob.glob(os.path.join(D, "**", "gradcam.zip"), recursive=True)
    if zips:
        zipfile.ZipFile(zips[0]).extractall("/kaggle/working")
        GC = "/kaggle/working/gradcam" if os.path.isdir("/kaggle/working/gradcam") else "/kaggle/working"

REPO = os.environ.get("REPO_DIR", "/kaggle/working/repo")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, REPO], check=False)
TOOLS = os.path.join(REPO, "detection", "tools", "analysis")
if not os.path.isdir(TOOLS):
    TOOLS = D                                           # fallback: the copies uploaded with the data
sys.path.insert(0, TOOLS)
import failure_stats as fs, plot_tsne as pt
print("data:", D, "| helpers:", TOOLS, "| grad-cam:", GC if os.path.isdir(GC) else "not found")

In [ ]:
# 2) Load every model's per-object table (+ its 512-d encoder features)
ORDER = ["random_init_on_radar", "radar_scratch", "radar_ft_from_livox", "livox_e20_on_radar",
         "random_init_on_livox", "livox_pretrain"]
M = {}
for f in sorted(glob.glob(os.path.join(D, "feats_*.npz"))):
    z = dict(np.load(f, allow_pickle=False))
    z["name"], z["sensor"] = str(z["name"]), str(z["sensor"])
    z["class_names"] = [str(c) for c in z["class_names"]]
    M[z["name"]] = z
order = [n for n in ORDER if n in M] + [n for n in M if n not in ORDER]
ref = M[order[0]]
for n in order:
    assert np.array_equal(M[n]["frame"], ref["frame"]) and np.array_equal(M[n]["cls"], ref["cls"]), n + ": different objects"

CLS = np.array([ref["class_names"][c - 1] for c in ref["cls"]])
SEQ = ref["sequence"]
df = pd.concat([pd.DataFrame({"model": n, "sensor": M[n]["sensor"], "obj": np.arange(len(CLS)), "cls": CLS,
                              "weather": M[n]["weather"], "light": M[n]["light"], "sequence": SEQ,
                              "dist": M[n]["dist"], "npts": M[n]["npts"], "detected": M[n]["detected"].astype(bool),
                              "score": M[n]["score"], "iou": M[n]["iou"]}) for n in order], ignore_index=True)
RADAR = [n for n in order if M[n]["sensor"] == "radar"]
print(len(CLS), "objects |", pd.Series(CLS).value_counts().to_dict(), "| models:", order)

## Part 1 — What happened
Recall = fraction of labelled objects the full model detected (same IoU thresholds as the AP evaluation).
Pedestrians and bikes are nearly invisible to radar, so the **vehicle** rows carry the transfer story.

In [ ]:
display(df.groupby(["model", "cls"])["detected"].mean().unstack().loc[order].round(3))
veh = df[df.cls == "Vehicle"]
display(veh.groupby(["model", "weather"])["detected"].mean().unstack().loc[order].round(3))
display(veh.groupby(["model", "light"])["detected"].mean().unstack().loc[order].round(3))

## Part 2 — Is the transfer gain real? (paired tests on the same objects)
McNemar = objects only one model detects; CI = resampling whole sequences; subgroup rows marked `*` survive Holm
correction; the regressions say what makes vehicles hard and where the fine-tuned model gains most.
This covers variation in the **data**, not in training — seeds are still needed to generalise.

In [ ]:
if {"radar_scratch", "radar_ft_from_livox"} <= set(M):
    out = subprocess.run([sys.executable, os.path.join(TOOLS, "failure_stats.py"),
                          "--baseline", os.path.join(D, "feats_radar_scratch.npz"),
                          "--treatment", os.path.join(D, "feats_radar_ft_from_livox.npz"),
                          "--out_dir", "stats_out"], capture_output=True, text=True)
    print(out.stdout[-12000:] or out.stderr[-3000:])

## Part 3 — Where does transfer help or hurt?
Vehicle recall against how much radar evidence an object has (points inside its box) and distance.
A gap between `radar_ft_from_livox` and `radar_scratch` that is largest at **few points / long range** means the
LiDAR prior fills in for missing radar evidence. `livox_e20_on_radar` (no fine-tuning) shows the raw gap.

In [ ]:
NPTS_BINS = [0, 1, 3, 5, 10, 20, 50, 10 ** 9]
DIST_BINS = [0, 10, 20, 30, 40, 50, 60, 80]
fig, axes = plt.subplots(1, 3, figsize=(19, 4.8))
for n in RADAR:
    d = veh[veh.model == n]
    for ax, col, bins in [(axes[0], "npts", NPTS_BINS), (axes[1], "dist", DIST_BINS)]:
        r = d.groupby(pd.cut(d[col], bins, right=False), observed=False)["detected"].mean()
        ax.plot(range(len(r)), r.values, marker="o", label=n)
        ax.set_xticks(range(len(r))); ax.set_xticklabels([str(i) for i in r.index], rotation=30)
    w = d.groupby("weather")["detected"].mean()
    axes[2].plot(range(len(w)), w.values, marker="o", label=n)
    axes[2].set_xticks(range(len(w))); axes[2].set_xticklabels(w.index, rotation=30)
for ax, t in zip(axes, ["vehicle recall vs radar points in box", "vehicle recall vs distance (m)",
                        "vehicle recall by weather"]):
    ax.set_title(t); ax.set_ylim(0, 1); ax.grid(alpha=.3)
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()
print("vehicles per radar-points bin:", veh[veh.model == RADAR[0]].groupby(
      pd.cut(veh[veh.model == RADAR[0]].npts, NPTS_BINS, right=False), observed=False).size().to_dict())

## Part 4 — What did each encoder learn? (probes vs the random-weights floor)
kNN accuracy / silhouette: how separable the classes are in the encoder's features. `knn_weather_acc`: how much the
weather leaks into object features (lower = more weather-robust). Compare each model with the random floor of the
same input: only the difference is learned.

In [ ]:
rows = {}
for n in order:
    idx = pt.balanced_subset(M[n]["cls"], 1500)
    rows[n], _ = pt.metrics(M[n], idx)
probe = pd.DataFrame(rows).T[["knn_class_acc", "silhouette_class", "knn_weather_acc", "recall_all"]]
floor = {"radar": "random_init_on_radar", "livox": "random_init_on_livox"}
probe["knn_above_random"] = [probe.loc[n, "knn_class_acc"] - probe.loc[floor[M[n]["sensor"]], "knn_class_acc"]
                             if floor.get(M[n]["sensor"]) in probe.index else np.nan for n in probe.index]
display(probe.round(3))

## Part 5 — How similar are the representations? (linear CKA, 0 = unrelated, 1 = identical up to rotation/scale)
Key pairs:
* `livox_pretrain` vs `livox_e20_on_radar` — **same weights, different input**: the modality gap inside the encoder.
* `radar_ft_from_livox` vs `livox_e20_on_radar` — how much fine-tuning **kept** of the pretrained radar description.
* `radar_ft_from_livox` vs `livox_pretrain` — does the fine-tuned radar encoder still describe objects **like the
  LiDAR model does with LiDAR**? High = cross-modal alignment survived.
* `radar_ft_from_livox` vs `radar_scratch` — did fine-tuning end up where training from scratch ends up anyway?

In [ ]:
def linear_cka(X, Y):
    X = X - X.mean(0); Y = Y - Y.mean(0)
    return np.linalg.norm(X.T @ Y) ** 2 / (np.linalg.norm(X.T @ X) * np.linalg.norm(Y.T @ Y))

idx = pt.balanced_subset(ref["cls"], 1000)
F = {n: M[n]["feat"][idx].astype(np.float64) for n in order}
K = pd.DataFrame([[linear_cka(F[a], F[b]) for b in order] for a in order], index=order, columns=order)
fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(K.values, vmin=0, vmax=1, cmap="viridis")
ax.set_xticks(range(len(order))); ax.set_xticklabels(order, rotation=45, ha="right")
ax.set_yticks(range(len(order))); ax.set_yticklabels(order)
for i in range(len(order)):
    for j in range(len(order)):
        ax.text(j, i, "%.2f" % K.values[i, j], ha="center", va="center", color="w" if K.values[i, j] < .6 else "k")
fig.colorbar(im); ax.set_title("linear CKA between encoders (same objects)"); plt.tight_layout(); plt.show()

## Part 6 — Does looking "LiDAR-like" help a radar encoder detect? (tests the case for cross-modal distillation)
For each radar model, a ridge regression predicts the LiDAR model's feature of an object (`livox_pretrain`, LiDAR
input) from the radar feature of the **same** object, trained and tested on different sequences. The per-object
cosine between prediction and truth = how well that object's radar description aligns with its LiDAR description.
A logistic regression then asks: **does alignment predict detection, beyond the number of radar points and distance?**
If yes, training the radar encoder to match LiDAR features on paired frames (distillation) is a promising next step.

In [ ]:
import statsmodels.api as sm
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

TEACHER = "livox_pretrain"
vmask = CLS == "Vehicle"
if TEACHER in M:
    Y = StandardScaler().fit_transform(M[TEACHER]["feat"][vmask].astype(np.float64))
    groups = SEQ[vmask]
    summary = []
    for n in [m for m in RADAR if not m.startswith("random")]:
        X = StandardScaler().fit_transform(M[n]["feat"][vmask].astype(np.float64))
        pred = np.zeros_like(Y)
        for tr, te in GroupKFold(n_splits=5).split(X, Y, groups):
            pred[te] = Ridge(alpha=100.0).fit(X[tr], Y[tr]).predict(X[te])
        align = (pred * Y).sum(1) / (np.linalg.norm(pred, axis=1) * np.linalg.norm(Y, axis=1) + 1e-9)
        det = M[n]["detected"][vmask].astype(int)
        npts = M[n]["npts"][vmask]
        exog = sm.add_constant(pd.DataFrame({"alignment_z": (align - align.mean()) / align.std(),
                                             "log2_npts": np.log2(1 + npts), "dist_10m": M[n]["dist"][vmask] / 10}))
        try:
            res = sm.GLM(det, exog, family=sm.families.Binomial()).fit(
                cov_type="cluster", cov_kwds={"groups": pd.factorize(groups)[0]})
            orr, p = np.exp(res.params["alignment_z"]), res.pvalues["alignment_z"]
        except Exception as e:
            orr, p = np.nan, np.nan
        summary.append(dict(model=n, mean_alignment=align.mean(), recall=det.mean(),
                            odds_ratio_per_sd_alignment=orr, p=p))
        df.loc[(df.model == n) & (df.cls == "Vehicle"), "alignment"] = align
    display(pd.DataFrame(summary).round(4))

## Part 7 — The modality gap, seen directly (same weights, LiDAR vs radar input)
Joint t-SNE of `livox_pretrain` (LiDAR input) and `livox_e20_on_radar` (radar input) vehicle features. Two separate
islands = the encoder describes the same cars completely differently per sensor (input/domain gap). Overlap = the
encoder already maps both sensors to a shared description.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
A, B = "livox_pretrain", "livox_e20_on_radar"
if {A, B} <= set(M):
    rng = np.random.default_rng(0)
    vi = rng.choice(np.flatnonzero(vmask), size=min(1500, vmask.sum()), replace=False)
    Z = StandardScaler().fit_transform(np.vstack([M[A]["feat"][vi], M[B]["feat"][vi]]).astype(np.float64))
    emb = TSNE(2, perplexity=30, init="pca", random_state=0).fit_transform(PCA(50, random_state=0).fit_transform(Z))
    k = len(vi)
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].scatter(*emb[:k].T, s=3, label="LiDAR input"); axes[0].scatter(*emb[k:].T, s=3, label="radar input")
    axes[0].legend(markerscale=4); axes[0].set_title("same encoder, two sensors")
    det = np.r_[M[A]["detected"][vi], M[B]["detected"][vi]].astype(bool)
    axes[1].scatter(*emb[~det].T, s=3, c="tab:red", label="missed"); axes[1].scatter(*emb[det].T, s=3, c="tab:green", label="detected")
    axes[1].legend(markerscale=4); axes[1].set_title("detected vs missed")
    for ax in axes: ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout(); plt.show()

## Part 8 — Which objects did transfer fix, and which did it break?
Object by object: `radar_scratch` vs `radar_ft_from_livox`. If "fixed" objects have few points / long range, the
LiDAR prior is compensating for weak radar evidence; if "broken" ones cluster in one weather, look there with Grad-CAM.

In [ ]:
if {"radar_scratch", "radar_ft_from_livox"} <= set(M):
    a = df[df.model == "radar_scratch"].reset_index(drop=True)
    b = df[df.model == "radar_ft_from_livox"].reset_index(drop=True)
    a["outcome"] = np.select([~a.detected & b.detected, a.detected & ~b.detected, a.detected & b.detected],
                             ["fixed by transfer", "broken by transfer", "both detect"], "both miss")
    display(a.groupby("outcome").agg(n=("cls", "size"), vehicle_share=("cls", lambda s: (s == "Vehicle").mean()),
                                     median_radar_pts=("npts", "median"), median_dist=("dist", "median")).round(2))
    display(pd.crosstab(a.outcome, a.weather, normalize="columns").round(3))

## Part 9 — Grad-CAM, same frames side by side
Bright = where the features that raised this object's heatmap score are. On the object = evidence-driven; on clutter
or empty space = misled / context-driven. Change `WEATHER` to look elsewhere (the folders that exist are printed).

In [ ]:
WEATHER = "Fog"
if os.path.isdir(GC):
    print("weathers available:", sorted(os.listdir(GC)))
    files = glob.glob(os.path.join(GC, WEATHER, "*.png"))
    by_frame = {}
    for f in files:
        name, frame = os.path.basename(f)[:-4].rsplit("_" + WEATHER + "_", 1)
        by_frame.setdefault(frame, {})[name] = f
    for frame, per_model in sorted(by_frame.items()):
        print("=== frame", frame, "| models:", sorted(per_model))
        for n in [m for m in order if m in per_model]:
            print(n)
            if Image is not None:
                display(Image(per_model[n]))

## Reading the evidence → what to try next for better transfer

| If you see … | it suggests … | next experiment |
|---|---|---|
| `knn_above_random` small for the radar models | radar encoders learn little beyond geometry | stronger radar input: accumulate several radar sweeps (`SWEEP_COUNT`) |
| Part 7: two separate islands, low CKA(livox_pretrain, livox_e20_on_radar) | the gap is mostly at the **input** (density, 4th channel) | pretrain on **radar-like LiDAR**: subsample LiDAR to radar density, drop or randomise the 4th channel |
| Part 3/8: transfer gains concentrated at few points / long range | the LiDAR prior compensates for missing radar evidence | push it: sparser LiDAR augmentation during pretraining; longer pretraining |
| Part 6: alignment significantly predicts detection | LiDAR-like radar features detect better | **cross-modal distillation**: add a loss matching radar features to LiDAR features on paired DSERT frames |
| Part 5: CKA(ft, scratch) high, CKA(ft, livox_pretrain) low | fine-tuning **overwrote** the prior | lower encoder learning rate / freeze early layers (`--fix_pretrained_weights`), `PRETRAINED_SCOPE=all` |
| Pedestrian / bike recall ≈ 0 everywhere | radar barely observes them | report vehicles separately; these classes need another sensor |

Whatever you find: one training seed per model, so confirm the promising ones with repeated seeds before claiming them.